In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import WeightedRandomSampler, Dataset, DataLoader
from torchvision import transforms, models

import numpy as np
import pyarrow.parquet as pq
import pandas as pd 
from pathlib import Path
from PIL import Image
from sklearn.preprocessing import StandardScaler

In [52]:
PARQUET_PATH = Path("/Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_train.parquet")
PHOTO_PATH = Path("/Users/mainframe/Workspace/Graduate/dataset/train/photos")
PARQUET_VAL_PATH = Path("/Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_validate.parquet")
PHOTO_VAL_PATH = Path("/Users/mainframe/Workspace/Graduate/dataset/validate/photos")

df = pq.read_table(source=PARQUET_PATH, use_threads=True).to_pandas()
df_val = pq.read_table(source=PARQUET_VAL_PATH, use_threads=True).to_pandas()

scaler = StandardScaler()
cols_to_scale = ['temp', 'humid', 'co2', 'light', 'stem_area', 'leaf_area']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
df_val[cols_to_scale] = scaler.fit_transform(df_val[cols_to_scale])

In [55]:
class MyPipeline:
    def __init__(self, size=224):
        # This is our 'Transformation Contract'
        self.transform = transforms.Compose([
            # 1. Resize to the target ResNet size
            transforms.Resize((size, size)),
            
            # 2. Geometric Augmentations
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            
            # 3. Color/Lighting Augmentations
            transforms.ColorJitter(
                brightness=0.2, 
                contrast=0.2, 
                saturation=0.2, 
                hue=0.1
            ),
            
            # 4. Simulate sensor noise/focus issues
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=(5, 5), sigma=(0.1, 2.0))
            ], p=0.3),
            
            # 5. The essentials for ResNet
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]
            )
        ])

        self.val_transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def process(self, image_path: Path):
        """Reads and transforms a single image for the M4 Max NPU."""
        try:
            # 1. Open the image
            img = Image.open(image_path).convert('RGB')
            
            # 2. Apply the transformations
            img_tensor = self.transform(img)
            
            # 3. Add a batch dimension (C, H, W) -> (1, C, H, W)
            # Neural networks expect a 'batch' even if it's just one image
            return img_tensor.unsqueeze(0)
            
        except Exception as e:
            print(f"Failed to process {image_path.name}: {e}")
            return None

In [56]:
class MyDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = Path(img_dir)
        self.transform = transform
        
        # Prepare tabular features for the MLP branch
        # We normalize these so the model doesn't choke on large numbers
        self.tabular_cols = ['temp', 'humid', 'co2', 'light', 'stem_area', 'leaf_area']
        self.labels = self.df['delta_days'].values
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Get the row data
        row = self.df.iloc[idx]
        
        # 2. Handle the Image
        # Assuming your file_name needs an extension like .jpg
        img_path = self.img_dir / f"{row['file_name']}.jpg"
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        # 3. Handle the Tabular Data (MLP Input)
        # Convert the row features into a FloatTensor
        tab_data = torch.tensor(row[self.tabular_cols].values.astype('float32'))
        
        # 4. Handle the Label (Target)
        # Convert 'growth' (1, 2, 3) to a float for Regression
        label = torch.tensor([float(row['growth'])])
        
        return image, tab_data, label

In [57]:
pipeline = MyPipeline(size=224)

train_ds = MyDataset(
    dataframe=df, 
    img_dir=PHOTO_PATH, 
    transform=pipeline.transform # Using data argumentation 
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)

val_ds = MyDataset(
    dataframe=df_val, 
    img_dir=PHOTO_VAL_PATH, 
    transform=pipeline.val_transform #use normal stuff
)

val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

In [58]:
class PlantaGotchiMultiModal(nn.Module):
    def __init__(self, num_tabular_features=6, pretrained=True):
        super(PlantaGotchiMultiModal, self).__init__()
        
        # 1. Vision Branch: Using ResNet18
        # We load weights to get that 'ImageNet' intuition about shapes and textures
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        
        # We need the feature vector, not the 1000-class output.
        # ResNet18's last layer before the FC is 224 dimensions.
        vision_feature_dim = self.backbone.fc.in_features
        
        # Remove the original classification head
        self.backbone.fc = nn.Identity() 
        
        # 2. Tabular Branch (Keeping your logic)
        self.tabular_branch = nn.Sequential(
            nn.Linear(num_tabular_features, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU()
        )
        
        # 3. The Fusion Head
        # input = 224 (Vision) + 16 (Tabular) = 528
        self.regressor = nn.Sequential(
            nn.Linear(vision_feature_dim + 16, 64),
            nn.ReLU(),
            nn.Dropout(0.2), # Adding a bit of regularization for the beefier model
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1) 
        )

    def forward(self, img, tab):
        # Image features: [Batch, 512]
        v_feat = self.backbone(img)
        
        # Tabular features: [Batch, 16]
        t_feat = self.tabular_branch(tab)
        
        # Glue 'em together: [Batch, 528]
        combined = torch.cat((v_feat, t_feat), dim=1)
        
        return self.regressor(combined)

In [59]:
# Initialization on M4 Max
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = PlantaGotchiMultiModal().to(device)
print(f"Model initialized on: {device}")

Model initialized on: mps


In [60]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [61]:
model.train() # Set to training mode
running_loss = 0.0

In [62]:
for batch_idx, (images, tabular, labels) in enumerate(train_loader):
    # Move everything to the NPU
    images = images.to(device)
    tabular = tabular.to(device)
    labels = labels.to(device).float() # Labels must be float for MSE

    # --- THE CORE MATH ---
    optimizer.zero_grad()               # Clear old gradients
    outputs = model(images, tabular)    # Forward pass: THE GUESS
    loss = criterion(outputs, labels)   # How wrong were we?
    loss.backward()                     # Backpropagation
    optimizer.step()                    # Update weights
    # ---------------------

    running_loss += loss.item()
    
    if batch_idx % 10 == 0:
        print(f"Batch {batch_idx} | Current Loss: {loss.item():.4f}")

print(f"Epoch finished with Average Loss: {running_loss / len(train_loader):.4f}")

Batch 0 | Current Loss: 4.0532
Batch 10 | Current Loss: 0.6550
Batch 20 | Current Loss: 0.6282
Batch 30 | Current Loss: 0.1945
Batch 40 | Current Loss: 0.2333
Batch 50 | Current Loss: 0.3114
Batch 60 | Current Loss: 0.1705
Batch 70 | Current Loss: 0.2248
Batch 80 | Current Loss: 0.1201
Batch 90 | Current Loss: 0.1209
Batch 100 | Current Loss: 0.1531
Batch 110 | Current Loss: 0.1529
Batch 120 | Current Loss: 0.0917
Batch 130 | Current Loss: 0.2255
Batch 140 | Current Loss: 0.3074
Batch 150 | Current Loss: 0.0886
Batch 160 | Current Loss: 0.0605
Batch 170 | Current Loss: 0.0735
Batch 180 | Current Loss: 0.1760
Batch 190 | Current Loss: 0.0824
Batch 200 | Current Loss: 0.0735
Batch 210 | Current Loss: 0.1453
Batch 220 | Current Loss: 0.1738
Batch 230 | Current Loss: 0.2455
Batch 240 | Current Loss: 0.1066
Batch 250 | Current Loss: 0.3299
Batch 260 | Current Loss: 0.1217
Batch 270 | Current Loss: 0.0834
Batch 280 | Current Loss: 0.1212
Batch 290 | Current Loss: 0.1792
Batch 300 | Current L

In [64]:
model.eval() # Set to evaluation mode
val_loss = 0.0
total_error = 0.0

with torch.no_grad(): # Disable gradient calculation
    for images, tabular, labels in val_loader:
        # Move to NPU
        images = images.to(device)
        tabular = tabular.to(device)
        labels = labels.to(device).float()

        # Forward pass ONLY
        outputs = model(images, tabular)
        
        # Calculate Loss
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        
        # Calculate MAE (Mean Absolute Error) 
        # This tells you how many 'stages' off you are on average
        total_error += torch.abs(outputs - labels).sum().item()

avg_val_loss = val_loss / len(val_loader)
avg_mae = total_error / len(val_ds)

print(f"Validation Loss: {avg_val_loss:.4f}")
print(f"Mean Absolute Error: {avg_mae:.2f} days")

Validation Loss: 0.2183
Mean Absolute Error: 0.41 days


In [66]:
stage_results = {}

for stage in range(0, 40):
    # 1. Filter the dataframe for this specific stage
    stage_df = df_val[df_val['delta_days'] == stage]
    
    # 2. Initialize a temporary dataset and loader for just this stage
    temp_ds = MyDataset(
        dataframe=stage_df,
        img_dir=PHOTO_VAL_PATH,
        transform=pipeline.transform
    )
    temp_loader = DataLoader(temp_ds, batch_size=16, shuffle=False, num_workers=0)
    
    stage_loss = 0.0
    stage_mae_sum = 0.0
    
    model.eval()
    with torch.no_grad():
        for images, tabular, labels in temp_loader: # Using temp_loader here!
            images, tabular, labels = images.to(device), tabular.to(device), labels.to(device).float()
            
            outputs = model(images, tabular)
            loss = criterion(outputs, labels)
            
            stage_loss += loss.item() * images.size(0)
            stage_mae_sum += torch.abs(outputs - labels).sum().item()
    
    # 3. Calculate actual Means
    total_samples = len(temp_ds)
    avg_loss = stage_loss / total_samples
    avg_mae = stage_mae_sum / total_samples
    
    print(f"--- Stage {stage} (n={total_samples}) ---")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Mean Absolute Error: {avg_mae:.2f} days\n")

--- Stage 0 (n=32) ---
Average Loss: 0.0909
Mean Absolute Error: 0.29 days

--- Stage 1 (n=28) ---
Average Loss: 0.0939
Mean Absolute Error: 0.30 days

--- Stage 2 (n=26) ---
Average Loss: 0.0993
Mean Absolute Error: 0.31 days

--- Stage 3 (n=28) ---
Average Loss: 0.1047
Mean Absolute Error: 0.31 days

--- Stage 4 (n=59) ---
Average Loss: 0.1006
Mean Absolute Error: 0.31 days

--- Stage 5 (n=56) ---
Average Loss: 0.1530
Mean Absolute Error: 0.34 days

--- Stage 6 (n=38) ---
Average Loss: 0.3796
Mean Absolute Error: 0.49 days

--- Stage 7 (n=49) ---
Average Loss: 0.1849
Mean Absolute Error: 0.38 days

--- Stage 8 (n=40) ---
Average Loss: 0.3746
Mean Absolute Error: 0.52 days

--- Stage 9 (n=49) ---
Average Loss: 0.3110
Mean Absolute Error: 0.47 days

--- Stage 10 (n=65) ---
Average Loss: 0.1442
Mean Absolute Error: 0.35 days

--- Stage 11 (n=50) ---
Average Loss: 0.1954
Mean Absolute Error: 0.39 days

--- Stage 12 (n=39) ---
Average Loss: 0.1475
Mean Absolute Error: 0.35 days

--- Stage